In [ ]:
from src.utils import DATA_DIR
print(DATA_DIR)


In [ ]:
from src.get_dataset_info import get_all_info
df_transcriptions, df_results, df_embeddings, df_features = get_all_info(transcribe=False, evaluate=False, embeddings=False, features= False)

In [ ]:
import os
import pandas as pd
df_results_without_label = df_results.drop(columns=["label"])

df = pd.merge(df_results_without_label, df_features, on="id", how="inner")
df = pd.merge(df, df_embeddings, on="id", how="inner")
df

In [ ]:
X = df.drop(columns=["label"])
y = df["label"]

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegressionCV

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

enet_cv = LogisticRegressionCV(
    penalty='elasticnet',
    solver='saga',
    l1_ratios=[0.1, 0.3, 0.5, 0.7, 0.9],
    Cs= np.logspace(-3, 1, 10),
    cv=5,
    scoring='f1',
    max_iter=5000,
    n_jobs=-1
)

enet_cv.fit(X_scaled, y)

print("Best C:", enet_cv.C_[0])
print("Best l1_ratio:", enet_cv.l1_ratio_[0])

importance_enet = np.mean(np.abs(enet_cv.coef_), axis=0)

df_enet = pd.DataFrame({
    "feature": X.columns,
    "score": importance_enet
}).sort_values("score", ascending=False)

df_enet[df_enet["score"] > 0]

features_enet = df_enet[df_enet["score"] > 0]["feature"].tolist()
print("Selected features:", features_enet)

In [ ]:
from src.utils import create_5cv
indices = df.index.tolist()
labels = df["label"].tolist()

dict_id_labels = dict(zip(indices, labels))
folds = create_5cv(dict_id_labels=dict_id_labels, val=True)

In [ ]:
import pandas as pd
X_data = X[features_enet].copy()
y_data = df["label"].copy()

models = {
    "SVM": (
        SVC(kernel="linear", class_weight="balanced", probability=True),
        {"C": [0.01, 0.1, 1, 10, 100]}
    ),

    "SVM-RBF": (
        SVC(kernel="rbf", class_weight="balanced", probability=True),
        {"C": [0.01, 0.1, 1, 10, 100]}
    ),

    "RandomForest": (
        RandomForestClassifier(class_weight="balanced", random_state=42),
        {"n_estimators": [100, 200], "max_depth": [5, 10, None]}
    ),

    "LogReg": (
        LogisticRegression(class_weight="balanced", max_iter=5000),
        {"C": [0.01, 0.1, 1, 10, 100]}
    ),

    "NaiveBayes": (
        GaussianNB(),
        {}  # sin hiperparámetros
    )
}

all_results = []

for model_name, (model, param_grid) in models.items():

    print("\n" + "="*60)
    print(f"MODEL: {model_name}")
    print("="*60)

    model_results = []

    for fold in folds:

        train_idx = X_data.index.isin(fold['train_ids'])
        val_idx = X_data.index.isin(fold['val_ids']) if fold['val_ids'] is not None else None
        test_idx = X_data.index.isin(fold['test_ids'])

        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]

        if len(param_grid) > 0:

            grid = GridSearchCV(
                model,
                param_grid=param_grid,
                cv=3,
                scoring="f1",
                n_jobs=-1
            )

            grid.fit(X_train, y_train)
            best_model = grid.best_estimator_

        else:
            best_model = model
            best_model.fit(X_train, y_train)

        y_pred = best_model.predict(X_test)

        if hasattr(best_model, "predict_proba"):
            y_prob = best_model.predict_proba(X_test)[:, 1]
        else:
            y_prob = best_model.decision_function(X_test)

        fold_metrics = {
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_prob)
        }

        model_results.append(fold_metrics)

    df_model = pd.DataFrame(model_results)

    print("\n=== RESULTS ===")
    print(df_model)

    print("\n=== MEAN ± STD ===")
    print(df_model.mean(numeric_only=True))
    print(df_model.std(numeric_only=True))

    all_results.append(df_model)

df_all = pd.concat(all_results)

print("\n==============================")
print("GLOBAL COMPARISON")
print("==============================")

print(df_all.groupby("model").mean(numeric_only=True).sort_values("f1", ascending=False))

In [ ]:
import pandas as pd
def print_nested_results(df_nested):
    results = {}
    if "model" in df_nested.columns:
        print("\n=== MODEL RESULTS ===")
        for model in df_nested["model"].unique():
            print(f"\nModelel: {model}")
            df_model = df_nested[df_nested["model"] == model]
            print(df_model[["fold", "accuracy", "precision", "recall", "f1", "auc"]])

            print("\n=== MEAN ± STD ===")
            metrics = ["accuracy", "precision", "recall", "f1", "auc"]
            results[model] = {}
            for metric in metrics:
                mean = df_model[metric].mean()
                std = df_model[metric].std()
                print(f"{metric.capitalize()}: {mean:.4f} ± {std:.4f}")
                results[model][metric] = f"{mean:.4f} ± {std:.4f}"
    else:
        print("\n=== FOLD RESULTS ===")
        print(df_nested[["fold", "accuracy", "precision", "recall", "f1", "auc"]])

        print("\n=== MEAN ± STD ===")
        metrics = ["accuracy", "precision", "recall", "f1", "auc"]
        for metric in metrics:
            mean = df_nested[metric].mean()
            std = df_nested[metric].std()
            print(f"{metric.capitalize()}: {mean:.4f} ± {std:.4f}") 

    df_results = pd.DataFrame(results).T
    return df_results

results = print_nested_results(df_all)
print(results)

In [ ]:
import pandas as pd
X_data = X[features_enet].copy()
y_data = df["label"].copy()

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

base_models = {

    "SVM": (
        SVC(kernel="linear", class_weight="balanced", probability=True),
        {"C": [0.01, 0.1, 1, 10, 100]}
    ),

    "SVM-RBF": (
        SVC(kernel="rbf", class_weight="balanced", probability=True),
        {"C": [0.01, 0.1, 1, 10, 100]}
    ),

    "RandomForest": (
        RandomForestClassifier(class_weight="balanced", random_state=42),
        {"n_estimators": [100, 200], "max_depth": [5, 10, None]}
    ),

    "LogReg": (
        LogisticRegression(class_weight="balanced", max_iter=5000),
        {"C": [0.01, 0.1, 1, 10, 100]}
    ),

    "LDA": (
        LinearDiscriminantAnalysis(),
        {}
    )
}

all_results = []
best_estimators_global = {}
best_params_results = []

for model_name, (model, param_grid) in base_models.items():

    print("\n" + "="*60)
    print(f"MODEL: {model_name}")
    print("="*60)

    model_results = []
    best_estimators_fold = []

    for fold in folds:

        train_idx = X_data.index.isin(fold['train_ids'])
        val_idx = X_data.index.isin(fold['val_ids']) if fold['val_ids'] is not None else None
        test_idx = X_data.index.isin(fold['test_ids'])

        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]

        if len(param_grid) > 0:

            grid = GridSearchCV(
                estimator=model,
                param_grid=param_grid,
                cv=5,
                scoring="f1",
                n_jobs=-1
            )

            grid.fit(X_train, y_train)

            best_model = grid.best_estimator_

            best_estimators_fold.append((grid.best_score_, best_model))

            best_params_results.append({
                "model": model_name,
                "fold": fold,
                "best_f1_cv": grid.best_score_,
                **grid.best_params_
            })

        else:
            best_model = model
            best_model.fit(X_train, y_train)
            best_estimators_fold.append((0, best_model))

        y_pred = best_model.predict(X_test)

        if hasattr(best_model, "predict_proba"):
            y_prob = best_model.predict_proba(X_test)[:, 1]
        else:
            y_prob = best_model.decision_function(X_test)

        model_results.append({
            "model": model_name,
            "fold": fold,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_prob)
        })

        print(f"\nFold {fold} - {model_name}")
        print({
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_prob)
        })

    best_estimators_fold.sort(key=lambda x: x[0], reverse=True)
    best_estimators_global[model_name] = best_estimators_fold[0][1]

    df_model = pd.DataFrame(model_results)
    all_results.append(df_model)

best_svm = best_estimators_global["SVM"]
best_svm_rbf = best_estimators_global["SVM-RBF"]
best_rf = best_estimators_global["RandomForest"]
best_lr = best_estimators_global["LogReg"]
best_lda = best_estimators_global["LDA"]

voting_model = VotingClassifier(
    estimators=[
        ("svm", best_svm),
        ("svm_rbf", best_svm_rbf),
        ("rf", best_rf),
        ("lr", best_lr),
        ("lda", best_lda)
    ],
    voting="soft"
)

stacking_model = StackingClassifier(
    estimators=[
        ("svm", best_svm),
        ("svm_rbf", best_svm_rbf),
        ("rf", best_rf),
        ("lr", best_lr),
        ("lda", best_lda)
    ],
    final_estimator=LogisticRegression(
        class_weight="balanced",
        max_iter=5000
    ),
    cv=5,
    passthrough=False,
    n_jobs=-1
)

ensemble_models = {
    "Voting_Optimized": voting_model,
    "Stacking_Optimized": stacking_model
}

for ensemble_name, ensemble_model in ensemble_models.items():

    print("\n" + "="*60)
    print(f"ENSEMBLE: {ensemble_name}")
    print("="*60)

    ensemble_results = []

    for fold, (train_idx, test_idx) in enumerate(
        outer_cv.split(X_data, y_data),
        1
    ):

        X_train = X_data.iloc[train_idx]
        X_test = X_data.iloc[test_idx]

        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]

        ensemble_model.fit(X_train, y_train)

        y_pred = ensemble_model.predict(X_test)
        y_prob = ensemble_model.predict_proba(X_test)[:, 1]

        ensemble_results.append({
            "model": ensemble_name,
            "fold": fold,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_prob)
        })

        print(f"\nFold {fold} - {model_name}")
        print({
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred),
            "recall": recall_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "auc": roc_auc_score(y_test, y_prob)
        })

    df_ensemble = pd.DataFrame(ensemble_results)
    all_results.append(df_ensemble)

df_all = pd.concat(all_results, ignore_index=True)

print("\n" + "="*60)
print("GLOBAL COMPARISON")
print("="*60)

global_results = (
    df_all
    .groupby("model")
    .mean(numeric_only=True)
    .sort_values("f1", ascending=False)
)

print(global_results)